# WC 2026 Final & Third-place Forecast

Simulates both medal matches: **Spain–Argentina** (final) and **France–England** (third place).

Attach **two** datasets:
- **`soccer-train`** — trained models
- **`soccer-data`** — match CSVs (**must include both semi-final results** in `wc2026_results.csv`)

GPU ON. Internet ON.

1. **Cell 1** — setup + stage models
2. **Cell 2** — smoke (`500` simulations per match)
3. **Cell 3** — production (`80,000` simulations per match)


In [ ]:
import base64
import io
import json
import os
import subprocess
import sys
import warnings
import zipfile
from pathlib import Path

import pandas as pd

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

REPO_ZIP_B64 = """..."""

PIP_PACKAGES = [
    "pandas>=2.0",
    "pyarrow>=14.0",
    "pyyaml>=6.0",
    "lightgbm>=4.0",
    "numpy>=1.24",
    "scipy>=1.10",
    "tqdm>=4.66",
    "torch>=2.0",
]

WORK = Path("/kaggle/working")
REPO = WORK / "WorldCupPredictor"
MODELS = WORK / "models"
DATA_ROOT: Path | None = None

REQUIRED_DATA_FILES = (
    "results.csv",
    "wc2026_results.csv",
    "former_names.csv",
    "fixtures.csv",
    "match_stats.csv",
    "understat_matches.parquet",
)


def install_dependencies() -> None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES]
    )


def extract_project() -> Path:
    if REPO.exists():
        return REPO
    WORK.mkdir(parents=True, exist_ok=True)
    payload = base64.b64decode(REPO_ZIP_B64)
    with zipfile.ZipFile(io.BytesIO(payload)) as zf:
        zf.extractall(WORK)
    if not REPO.exists():
        raise RuntimeError(f"Expected project at {REPO} after extract")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)]
    )
    return REPO


def discover_data_root() -> Path:
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.is_dir():
        raise FileNotFoundError(
            "/kaggle/input not found. Attach soccer-data and soccer-train datasets."
        )

    def score(root: Path) -> int:
        return sum(1 for name in REQUIRED_DATA_FILES if (root / name).exists())

    candidates: list[Path] = []
    seen: set[str] = set()

    def add(path: Path) -> None:
        key = str(path.resolve())
        if key in seen:
            return
        seen.add(key)
        candidates.append(path)

    for results_csv in kaggle_input.rglob("results.csv"):
        add(results_csv.parent)
    for item in sorted(kaggle_input.iterdir()):
        if item.is_dir():
            add(item)
            for sub in sorted(item.rglob("*")):
                if sub.is_dir():
                    add(sub)

    ranked = sorted(((score(path), path) for path in candidates), key=lambda x: (-x[0], str(x[1])))
    preferred = ("soccer-data", "soccer-data-dataset")
    for found_score, path in ranked:
        if found_score >= 3 and (
            path.name in preferred or any(part in preferred for part in path.parts)
        ):
            return path
    for found_score, path in ranked:
        if found_score >= 3:
            return path

    lines = ["Could not find soccer data files."]
    if not candidates:
        lines.append("/kaggle/input is empty. Attach soccer-data and soccer-train.")
    else:
        lines.append("/kaggle/input contents:")
        for path in candidates:
            files = sorted(p.name for p in path.iterdir() if p.is_file())
            lines.append(f"  {path}: {', '.join(files[:8])}")
    raise FileNotFoundError("\n".join(lines))


def verify_data() -> Path:
    global DATA_ROOT
    DATA_ROOT = discover_data_root()
    missing = [name for name in REQUIRED_DATA_FILES if not (DATA_ROOT / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing in {DATA_ROOT}: {missing}")
    print("Data OK:", DATA_ROOT)
    return DATA_ROOT


def stage_and_verify_models() -> Path:
    """Copy soccer-train artifacts into /kaggle/working/models/."""
    sys.path.insert(0, str(REPO / "src"))
    from worldcup_predictor.kaggle_paths import (
        find_kaggle_models_source,
        models_dir_is_complete,
        stage_models,
        verify_model_artifacts,
    )

    MODELS.mkdir(parents=True, exist_ok=True)
    if models_dir_is_complete(MODELS):
        print("Models OK (already staged):", MODELS)
        return MODELS

    source = find_kaggle_models_source()
    if source is None:
        raise FileNotFoundError(
            "Could not find soccer-train/output dataset with model files. "
            "Attach soccer-train or output (calibration.json, gbm_*.txt, nn_model.pt, bayesian.json)."
        )
    stage_models(source, MODELS)
    missing = verify_model_artifacts(MODELS)
    if missing:
        raise FileNotFoundError(f"Staging incomplete, missing: {missing}")
    print("Models staged from", source, "->", MODELS)
    return MODELS

FORECAST = MODELS / "wc2026_forecast_final.json"
FORECAST_REPORT = MODELS / "forecast_final_report.json"


def run_forecast(
    profile: str = "kaggle",
    n_sims: int | None = None,
    seed: int = 42,
) -> int:
    repo = extract_project()
    data_root = verify_data()
    stage_and_verify_models()
    cmd = [
        sys.executable,
        str(repo / "scripts" / "run_kaggle_forecast_final.py"),
        "--profile", profile,
        "--seed", str(seed),
        "--models-dir", str(MODELS),
        "--data-root", str(data_root),
    ]
    if n_sims is not None:
        cmd.extend(["--sims", str(n_sims)])
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Forecast failed with exit code {result.returncode}")
    return result.returncode


def show_results(top_n: int = 5) -> None:
    if not FORECAST.exists():
        print(f"No forecast yet at {FORECAST}")
        return
    forecast = json.loads(FORECAST.read_text(encoding="utf-8"))
    print(f"Simulations: {forecast.get('n_sims')}")
    print("\nTop champion probabilities:")
    for team, prob in sorted(
        forecast["champion_probs"].items(), key=lambda x: -x[1]
    )[:top_n]:
        print(f"  {team}: {prob:.1%}")
    print("\nMost likely bracket champion:", forecast["most_likely_bracket"]["champion"])
    print("Path frequency:", f"{forecast.get('most_likely_bracket_fraction', 0):.3%}")
    if FORECAST_REPORT.exists():
        report = json.loads(FORECAST_REPORT.read_text(encoding="utf-8"))
        print(
            f"\nElapsed: {report.get('elapsed_seconds')}s | "
            f"s/sim: {report.get('seconds_per_sim')} | Profile: {report.get('profile')}"
        )


def show_match_win_probs() -> None:
    if not FORECAST.exists():
        print(f"No forecast yet at {FORECAST}")
        return
    forecast = json.loads(FORECAST.read_text(encoding="utf-8"))
    rows = forecast.get("match_win_probs", [])
    if not rows:
        print("No match_win_probs in forecast.")
        return
    print("\nMatch win probabilities (knockout):")
    for row in rows:
        print(
            f"  {row['round']}: {row['home']} vs {row['away']} — "
            f"P(home)={row['p_home_win']:.1%}, P(away)={row['p_away_win']:.1%} "
            f"(n={row['n_sims']})"
        )


install_dependencies()
extract_project()
verify_data()
stage_and_verify_models()
print("Setup complete. Next: run_forecast(profile='fast', n_sims=500)")


In [ ]:
# Smoke test. Must pass before production run.
run_forecast(profile='fast', n_sims=500)
show_results()
show_match_win_probs()


In [ ]:
# Production final + third-place forecast — 80,000 simulations each.
run_forecast(profile='kaggle', n_sims=80_000)
show_results(top_n=10)
show_match_win_probs()
